In [1]:
import pygame
import random
import json
import math
from typing import List, Tuple, Dict, Any

#Класс частицы фейерверка
class Particle:

    # Создаем одну частицу с позицией, цветом и параметрами из конфигурации
    def __init__(self, x: float, y: float, color: Tuple[int, int, int], config: Dict[str, Any]):
        self.x = x
        self.y = y
        self.color = color
        self.config = config
        
        # Начальная скорость
        angle = random.uniform(0, 2 * math.pi)
        speed = random.uniform(
            config["particle_min_speed"], 
            config["particle_max_speed"]
        )
        self.vx = math.cos(angle) * speed
        self.vy = math.sin(angle) * speed
        
        # Время жизни
        self.lifetime = random.uniform(
            config["particle_min_lifetime"],
            config["particle_max_lifetime"]
        )
        self.max_lifetime = self.lifetime
        
        # Размер
        self.size = random.uniform(
            config["particle_min_size"],
            config["particle_max_size"]
        )

    #Обновление состояния частицы
    def update(self, dt: float) -> bool:
        
        # Применяем гравитацию
        self.vy += self.config["gravity"] * dt
        
        # Обновляем позицию
        self.x += self.vx * dt
        self.y += self.vy * dt
        
        # Уменьшаем время жизни
        self.lifetime -= dt
        
        # Возвращаем True если частица еще жива
        return self.lifetime > 0

    # Отрисовка частицы
    def draw(self, surface: pygame.Surface) -> None:

        if self.lifetime > 0:
            # Вычисляем прозрачность на основе оставшегося времени жизни
            alpha = int(255 * (self.lifetime / self.max_lifetime))
            
            # Создаем поверхность для частицы с прозрачностью
            particle_surface = pygame.Surface((int(self.size * 2), int(self.size * 2)), pygame.SRCALPHA)
            
            # Рисуем круг с градиентом прозрачности
            color_with_alpha = (*self.color, alpha)
            pygame.draw.circle(
                particle_surface,
                color_with_alpha,
                (int(self.size), int(self.size)),
                int(self.size)
            )
            
            # Рисуем частицу на основной поверхности
            surface.blit(particle_surface, (int(self.x - self.size), int(self.y - self.size)))


# Класс фейерверка
class Firework:
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        
        # Начальная позиция (внизу экрана)
        self.x = random.uniform(100, config["screen_width"] - 100)
        self.y = config["screen_height"]
        
        # Цвет фейерверка
        self.color = random.choice(config["firework_colors"])
        
        # Скорость взлета
        self.speed = random.uniform(
            config["firework_min_speed"],
            config["firework_max_speed"]
        )
        
        # Целевая высота взрыва
        self.target_y = random.uniform(
            config["screen_height"] * 0.2,
            config["screen_height"] * 0.6
        )
        
        self.particles: List[Particle] = []
        self.exploded = False

    # Обновление состояния фейерверка
    def update(self, dt: float) -> bool:
        
        if not self.exploded:
            # Двигаем фейерверк вверх
            self.y -= self.speed * dt
            
            # Проверяем, достигли ли целевой высоты
            if self.y <= self.target_y:
                self.explode()
                return True
        else:
            # Обновляем частицы
            self.particles = [p for p in self.particles if p.update(dt)]
            
            # Фейерверк мертв, когда все частицы исчезли
            return len(self.particles) > 0
        
        return True

    # Создание взрыва частиц
    def explode(self) -> None:
        self.exploded = True
        num_particles = random.randint(
            self.config["min_particles"],
            self.config["max_particles"]
        )
        
        for _ in range(num_particles):
            self.particles.append(Particle(self.x, self.y, self.color, self.config))

    #Отрисовка фейерверка
    def draw(self, surface: pygame.Surface) -> None:
        if not self.exploded:
            # Рисуем летящий фейерверк
            pygame.draw.circle(surface, self.color, (int(self.x), int(self.y)), 3)
            # Добавляем хвост
            pygame.draw.line(
                surface, 
                self.color, 
                (int(self.x), int(self.y)), 
                (int(self.x), int(self.y + 10)), 
                2
            )
        else:
            # Рисуем все частицы
            for particle in self.particles:
                particle.draw(surface)


# Менеджер для управления фейерверками
class FireworksManager:
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.fireworks: List[Firework] = []
        self.last_firework_time = 0

    # Обновление всех фейерверков
    def update(self, dt: float, current_time: float) -> None:
        # Генерация новых фейерверков
        if (current_time - self.last_firework_time > 
            random.uniform(self.config["min_spawn_delay"], self.config["max_spawn_delay"])):
            self.fireworks.append(Firework(self.config))
            self.last_firework_time = current_time
        
        # Обновление существующих фейерверков
        self.fireworks = [fw for fw in self.fireworks if fw.update(dt)]

    # Отрисовка всех фейерверков
    def draw(self, surface: pygame.Surface) -> None:
        for firework in self.fireworks:
            firework.draw(surface)


# Основной класс приложения
class FireworksApp:
    
    def __init__(self):
        pygame.init()
        
        # Загрузка конфигурации
        self.config = self.load_config()
        
        # Настройка экрана
        self.screen = pygame.display.set_mode(
            (self.config["screen_width"], self.config["screen_height"])
        )
        pygame.display.set_caption("Фейерверки")
        
        # Создание менеджера фейерверков
        self.fireworks_manager = FireworksManager(self.config)
        
        # Часы для контроля FPS
        self.clock = pygame.time.Clock()
        
        # Время
        self.running = True
    
    @staticmethod

    #Загрузка конфигурации из JSON файла
    def load_config() -> Dict[str, Any]:
        try:
            with open("config.json", "r", encoding="utf-8") as f:
                return json.load(f)
        except FileNotFoundError:
            # Хранит ширину/высоту экрана, цвета фейерверков, диапазоны скоростей, количества и свойств частиц, параметры гравитации и задержек
            default_config = {
                "screen_width": 800,
                "screen_height": 600,
                "background_color": [0, 0, 0],
                "firework_colors": [
                    [255, 0, 0],      # Красный
                    [0, 255, 0],      # Зеленый
                    [0, 0, 255],      # Синий
                    [255, 255, 0],    # Желтый
                    [255, 0, 255],    # Пурпурный
                    [0, 255, 255],    # Голубой
                    [255, 165, 0]     # Оранжевый
                ],
                "firework_min_speed": 200,
                "firework_max_speed": 300,
                "min_particles": 50,
                "max_particles": 150,
                "particle_min_speed": 50,
                "particle_max_speed": 200,
                "particle_min_lifetime": 1.0,
                "particle_max_lifetime": 3.0,
                "particle_min_size": 1,
                "particle_max_size": 4,
                "gravity": 98,
                "min_spawn_delay": 0.5,
                "max_spawn_delay": 2.0,
                "fps": 60
            }
            
            # Сохраняем конфигурацию по умолчанию
            with open("config.json", "w", encoding="utf-8") as f:
                json.dump(default_config, f, indent=4, ensure_ascii=False)
            
            return default_config

    # Обработка событий
    def handle_events(self) -> None:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    self.running = False
                elif event.key == pygame.K_SPACE:
                    # Добавление фейерверка по клику 
                    self.fireworks_manager.fireworks.append(Firework(self.config))

    # Главный цикл приложения
    def run(self) -> None:
        while self.running:
            dt = self.clock.tick(self.config["fps"]) / 1000.0  
            current_time = pygame.time.get_ticks() / 1000.0
            
            self.handle_events()
            
            # Обновление
            self.fireworks_manager.update(dt, current_time)
            
            # Отрисовка
            self.screen.fill(tuple(self.config["background_color"]))
            self.fireworks_manager.draw(self.screen)
            
            pygame.display.flip()
        
        pygame.quit()


if __name__ == "__main__":
    app = FireworksApp()
    app.run()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
